## Setup and Data Loading

In [20]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import matplotlib
matplotlib.use('Agg')          # use non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from pathlib import Path

# output directories
STATIC_DIR = Path('../outputs/visualizations/static')
INTERACTIVE_DIR = Path('../outputs/visualizations/interactive')
STATIC_DIR.mkdir(parents=True, exist_ok=True)
INTERACTIVE_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully')
print(f'matplotlib {matplotlib.__version__}, seaborn {sns.__version__}, plotly {plotly.__version__}')

Libraries loaded successfully
matplotlib 3.10.8, seaborn 0.13.2, plotly 6.7.0


In [3]:
# Load the cleaned movie dataset produced in Lab 9
DATA_PATH = Path('../data/processed/cleaned/movies_clean.csv')
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (126, 15)
Columns: ['id', 'title', 'release_date', 'release_year', 'genre', 'primary_genre', 'budget_usd', 'revenue_usd', 'vote_average', 'vote_count', 'popularity', 'original_language', 'overview', 'director', 'country']


,id,title,release_date,release_year,genre,primary_genre,budget_usd,revenue_usd,vote_average,vote_count,popularity,original_language,overview,director,country
0,1084242,Zootopia 2,2025-11-26,2025,Action,Action,132435616,632805764,7.42,78,171.7147,en,Officer Judy Hopps and Nick Wilde return to th...,NaN,NaN
1,126889,Alien: Covenant,2017-05-09,2017,Sci-Fi,Sci-Fi,123402796,148327117,8.42,78,49.8742,en,"The crew of the colony ship Covenant, bound fo...",NaN,NaN
2,1167307,David,2025-12-14,2025,Action,Action,88131705,387584965,8.16,78,62.9864,en,A sweeping epic retelling of the biblical stor...,NaN,NaN


In [4]:
# Quick overview of key numeric columns
df[['budget_usd','revenue_usd','vote_average','vote_count','popularity']].describe().round(2)

,budget_usd,revenue_usd,vote_average,vote_count,popularity
count,1.260000e+02,1.260000e+02,126.00,126.00,126.00
mean,1.211492e+08,4.138927e+08,7.00,137064.70,105.26
std,6.930321e+07,4.488346e+08,0.95,496339.98,91.80
min,6.000000e+06,7.861270e+06,5.15,11.00,49.40
25%,6.203778e+07,1.140460e+08,6.27,78.00,61.56
50%,1.242652e+08,2.838907e+08,6.96,78.00,77.89
75%,1.811516e+08,5.209145e+08,7.56,78.00,107.13
max,3.160000e+08,2.923706e+09,9.50,2700000.00,828.46


---
## matplotlib Static Plots

### Visualization Principles

| Question | Chart type |
|----------|------------|
| Which movies earned the most? | Horizontal **bar** chart |
| How has the average rating changed over time? | **Line** chart |
| Is there a relationship between budget and revenue? | **Scatter** plot |
| What does the rating distribution look like? | **Histogram** + KDE |
| How do ratings vary across genres? | **Box** plot |
| What correlates with what? | **Heatmap** |

In [5]:
# Apply seaborn theme globally (Part 3 of the lecture – Styling)
sns.set_theme(style='whitegrid')
sns.set_context('notebook')
sns.set_palette('viridis')

### Bar Chart – Top 10 Movies by Revenue

We use `ax.barh()` (horizontal bar) because the movie titles are long strings.  
The **object-oriented API** (`fig, ax = plt.subplots()`) is preferred per the lecture material.

In [6]:
top10 = (df[df['revenue_usd'] > 0]
           .nlargest(10, 'revenue_usd')[['title','revenue_usd']]
           .sort_values('revenue_usd'))

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('viridis', 10)
bars = ax.barh(top10['title'], top10['revenue_usd'] / 1e9, color=colors)

for bar in bars:
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.2f}B', va='center', fontsize=9)

ax.set_xlabel('Box-Office Revenue (USD Billion)', fontsize=11)
ax.set_title('Top 10 Movies by Revenue', fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}B'))
ax.set_xlim(0, top10['revenue_usd'].max() / 1e9 * 1.18)
fig.tight_layout()

out_png = STATIC_DIR / 'top_movies_by_revenue.png'
out_pdf = STATIC_DIR / 'top_movies_by_revenue.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()
print(f'Saved → {out_png}  and  {out_pdf}')

Saved → ../outputs/visualizations/static/top_movies_by_revenue.png  and  ../outputs/visualizations/static/top_movies_by_revenue.pdf


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/1979659259.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Line Chart – Average Rating Over Years

Dual-axis line + bar chart: bars show the number of movies per year, the line shows the average vote rating.

In [7]:
yearly = (df.groupby('release_year')
            .agg(avg_rating=('vote_average','mean'), movie_count=('title','count'))
            .reset_index()
            .query('release_year >= 1980 and release_year <= 2025'))

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(yearly['release_year'], yearly['movie_count'], color='#a8d5e2', alpha=0.5, label='Movie Count')
ax1.set_ylabel('Number of Movies', color='#a8d5e2', fontsize=11)
ax1.tick_params(axis='y', labelcolor='#a8d5e2')

ax2 = ax1.twinx()
ax2.plot(yearly['release_year'], yearly['avg_rating'],
         color='#1a6faf', linewidth=2.5, marker='o', markersize=5, label='Avg Rating')
ax2.set_ylabel('Average Vote Rating (0–10)', color='#1a6faf', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#1a6faf')
ax2.set_ylim(0, 10)

ax1.set_xlabel('Release Year', fontsize=11)
ax1.set_title('Movie Releases and Average Rating Over the Years', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'avg_rating_over_years.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'avg_rating_over_years.pdf', bbox_inches='tight')
plt.show()
print('Saved → avg_rating_over_years  (PNG + PDF)')

Saved → avg_rating_over_years  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/1855495309.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Scatter Plot – Budget vs Revenue

In [8]:
data = df[(df['budget_usd'] > 0) & (df['revenue_usd'] > 0)].copy()
data['budget_M']  = data['budget_usd']  / 1e6
data['revenue_M'] = data['revenue_usd'] / 1e6

genres = data['primary_genre'].value_counts().nlargest(6).index.tolist()
data['genre_label'] = data['primary_genre'].where(data['primary_genre'].isin(genres), other='Other')

fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(data=data, x='budget_M', y='revenue_M',
                hue='genre_label', palette='viridis',
                size='vote_average', sizes=(30, 250), alpha=0.75, ax=ax)
max_v = max(data['budget_M'].max(), data['revenue_M'].max()) * 1.05
ax.plot([0, max_v], [0, max_v], '--', color='gray', linewidth=1.2, label='Break-even')

ax.set_xlabel('Production Budget (USD Million)', fontsize=11)
ax.set_ylabel('Box-Office Revenue (USD Million)', fontsize=11)
ax.set_title('Budget vs Revenue – Coloured by Genre', fontsize=14, fontweight='bold')
ax.legend(title='Genre / Size=Rating', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'budget_vs_revenue_scatter.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'budget_vs_revenue_scatter.pdf', bbox_inches='tight')
plt.show()
print('Saved → budget_vs_revenue_scatter  (PNG + PDF)')

Saved → budget_vs_revenue_scatter  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/3031227628.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## seaborn Statistical Plots

seaborn is built on matplotlib. `set_theme()`, `set_context()`, `set_palette()` provide consistent styling.

### Histogram + KDE – Rating Distribution

In [9]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['vote_average'].dropna(), bins=20, kde=True,
             color='#1a6faf', edgecolor='white', ax=ax)
ax.axvline(df['vote_average'].mean(), color='firebrick', linestyle='--', linewidth=1.8,
           label=f"Mean = {df['vote_average'].mean():.2f}")
ax.axvline(df['vote_average'].median(), color='darkorange', linestyle='-.', linewidth=1.8,
           label=f"Median = {df['vote_average'].median():.2f}")
ax.set_xlabel('Vote Average (0–10)', fontsize=11)
ax.set_ylabel('Number of Movies', fontsize=11)
ax.set_title('Distribution of Movie Ratings', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'rating_distribution.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'rating_distribution.pdf', bbox_inches='tight')
plt.show()
print('Saved → rating_distribution  (PNG + PDF)')

Saved → rating_distribution  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/4010667295.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Box Plot – Ratings by Genre

In [10]:
order = (df.groupby('primary_genre')['vote_average']
           .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='primary_genre', y='vote_average',
            order=order, hue='primary_genre', palette='viridis', legend=False, ax=ax)
ax.set_xlabel('Genre', fontsize=11)
ax.set_ylabel('Vote Average (0–10)', fontsize=11)
ax.set_title('Movie Rating Distribution by Genre', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
fig.tight_layout()

fig.savefig(STATIC_DIR / 'genre_rating_boxplot.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'genre_rating_boxplot.pdf', bbox_inches='tight')
plt.show()
print('Saved → genre_rating_boxplot  (PNG + PDF)')

Saved → genre_rating_boxplot  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/2685904178.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Correlation Heatmap – seaborn

In [11]:
numeric_cols = ['budget_usd','revenue_usd','vote_average','vote_count','popularity']
corr = df[numeric_cols].corr()
corr.index = corr.columns = ['Budget','Revenue','Avg Rating','Vote Count','Popularity']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax, annot_kws={'size': 10})
ax.set_title('Correlation Matrix – Budget, Revenue, Rating, Popularity',
             fontsize=13, fontweight='bold')
fig.tight_layout()

fig.savefig(STATIC_DIR / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'correlation_heatmap.pdf', bbox_inches='tight')
plt.show()
print('Saved → correlation_heatmap  (PNG + PDF)')

Saved → correlation_heatmap  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/4273882639.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Subplot Layout

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Movie Industry Analytics Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Panel A – top 10 revenue
top = (df[df['revenue_usd'] > 0]
       .nlargest(10, 'revenue_usd')[['title','revenue_usd']]
       .sort_values('revenue_usd'))
axes[0,0].barh(top['title'], top['revenue_usd']/1e9, color=sns.color_palette('viridis', 10))
axes[0,0].set_title('Top 10 by Revenue', fontsize=11, fontweight='bold')
axes[0,0].set_xlabel('Revenue (USD B)')

# Panel B – rating distribution
sns.histplot(df['vote_average'].dropna(), bins=15, kde=True, color='#1a6faf', ax=axes[0,1])
axes[0,1].set_title('Rating Distribution', fontsize=11, fontweight='bold')

# Panel C – genre count
counts = df['primary_genre'].value_counts().head(8)
axes[1,0].bar(counts.index, counts.values, color=sns.color_palette('viridis', len(counts)))
axes[1,0].set_title('Movies per Genre', fontsize=11, fontweight='bold')
axes[1,0].tick_params(axis='x', rotation=30)

# Panel D – budget vs revenue
scatter_d = df[(df['budget_usd'] > 0) & (df['revenue_usd'] > 0)].copy()
sc = axes[1,1].scatter(scatter_d['budget_usd']/1e6, scatter_d['revenue_usd']/1e6,
                       c=scatter_d['vote_average'], cmap='viridis', alpha=0.65, s=40)
fig.colorbar(sc, ax=axes[1,1], label='Rating')
axes[1,1].set_title('Budget vs Revenue', fontsize=11, fontweight='bold')
axes[1,1].set_xlabel('Budget (M)')
axes[1,1].set_ylabel('Revenue (M)')

fig.tight_layout()
fig.savefig(STATIC_DIR / 'dashboard_subplots.png', dpi=300, bbox_inches='tight')
fig.savefig(STATIC_DIR / 'dashboard_subplots.pdf', bbox_inches='tight')
plt.show()
print('Saved → dashboard_subplots  (PNG + PDF)')

Saved → dashboard_subplots  (PNG + PDF)


/var/folders/1r/m0k0dpq16vd05wsn453djbp40000gn/T/ipykernel_71866/741816127.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Plotly Express Interactive Charts

Plotly produces **interactive web charts** with hover tooltips, zoom, pan, and legend toggle — all without extra code.

### Interactive Scatter – Budget vs Revenue

In [13]:
data = df[(df['budget_usd'] > 0) & (df['revenue_usd'] > 0)].copy()
data['budget_M']  = (data['budget_usd']  / 1e6).round(1)
data['revenue_M'] = (data['revenue_usd'] / 1e6).round(1)
data['roi'] = ((data['revenue_usd'] - data['budget_usd']) / data['budget_usd'] * 100).round(1)

fig = px.scatter(
    data, x='budget_M', y='revenue_M',
    color='primary_genre', size='vote_count',
    hover_name='title',
    hover_data={'release_year': True, 'vote_average': ':.2f', 'roi': ':.1f'},
    labels={'budget_M': 'Budget (USD M)', 'revenue_M': 'Revenue (USD M)'},
    title='Budget vs Box-Office Revenue – Interactive Explorer',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)

# break-even line
max_v = max(data['budget_M'].max(), data['revenue_M'].max()) * 1.05
fig.add_trace(go.Scatter(x=[0, max_v], y=[0, max_v], mode='lines',
                         line=dict(dash='dash', color='gray', width=1),
                         name='Break-even', hoverinfo='skip'))

fig.update_layout(legend_title='Genre', font=dict(family='Inter', size=13), height=580)

html_path = INTERACTIVE_DIR / 'budget_vs_revenue_interactive.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/budget_vs_revenue_interactive.html


### Interactive Bar – Top 10 by Popularity

In [14]:
top_pop = df.nlargest(10, 'popularity')[['title','popularity','revenue_usd','vote_average','release_year','primary_genre']].copy()
top_pop['revenue_M'] = (top_pop['revenue_usd'] / 1e6).round(1)

fig = px.bar(
    top_pop.sort_values('popularity', ascending=True),
    x='popularity', y='title', orientation='h',
    color='primary_genre',
    hover_name='title',
    hover_data={'release_year': True, 'vote_average': ':.2f', 'revenue_M': ':.1f'},
    labels={'popularity': 'TMDB Popularity Score', 'title': 'Movie'},
    title='Top 10 Movies by TMDB Popularity Score',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.update_layout(legend_title='Genre', font=dict(family='Inter', size=13), height=500)

html_path = INTERACTIVE_DIR / 'top_movies_popularity_bar.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/top_movies_popularity_bar.html


### Interactive Line – Movies per Year

In [15]:
yearly = (df.query('release_year >= 1980 and release_year <= 2025')
            .groupby('release_year')
            .agg(movie_count=('title','count'), avg_rating=('vote_average','mean'),
                 total_revenue_B=('revenue_usd', lambda s: (s.sum()/1e9).round(2)))
            .reset_index())

fig = px.line(yearly, x='release_year', y='movie_count', markers=True,
              hover_data={'avg_rating': ':.2f', 'total_revenue_B': ':.2f'},
              labels={'release_year': 'Year', 'movie_count': 'Number of Movies'},
              title='Movies Released per Year (1980–2025)',
              template='plotly_white')
fig.update_traces(line_color='#1a6faf', line_width=2.5, marker=dict(size=7))
fig.update_layout(font=dict(family='Inter', size=13), height=450)

html_path = INTERACTIVE_DIR / 'movies_per_year_line.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/movies_per_year_line.html


### Interactive Box – Genre Ratings

In [16]:
order = df.groupby('primary_genre')['vote_average'].median().sort_values(ascending=False).index.tolist()

fig = px.box(
    df, x='primary_genre', y='vote_average',
    category_orders={'primary_genre': order},
    color='primary_genre',
    hover_name='title',
    hover_data={'release_year': True, 'vote_average': ':.2f'},
    labels={'primary_genre': 'Genre', 'vote_average': 'Vote Average (0–10)'},
    title='Movie Rating Distribution by Genre',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid,
)
fig.update_layout(showlegend=False, font=dict(family='Inter', size=13), height=500)

html_path = INTERACTIVE_DIR / 'genre_rating_boxplot_interactive.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/genre_rating_boxplot_interactive.html


---
### Multi-Chart Plotly Layout (2×2 Dashboard)

In [17]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top 10 Movies by Revenue','Rating Distribution',
                    'Movies per Genre','Budget vs Revenue'),
    vertical_spacing=0.14, horizontal_spacing=0.10,
)

# 1 – revenue bar
t10 = df[df['revenue_usd'] > 0].nlargest(10,'revenue_usd').sort_values('revenue_usd')
fig.add_trace(go.Bar(x=t10['revenue_usd']/1e9, y=t10['title'], orientation='h',
                     marker_color='#1a6faf', name='Revenue',
                     hovertemplate='%{y}<br>$%{x:.2f}B<extra></extra>'), row=1, col=1)

# 2 – rating histogram
fig.add_trace(go.Histogram(x=df['vote_average'].dropna(), nbinsx=20,
                           marker_color='#2ca02c', name='Ratings',
                           hovertemplate='Rating: %{x}<br>Count: %{y}<extra></extra>'), row=1, col=2)

# 3 – genre count
gc = df['primary_genre'].value_counts()
fig.add_trace(go.Bar(x=gc.index, y=gc.values, marker_color='#ff7f0e', name='Genres',
                     hovertemplate='%{x}: %{y} movies<extra></extra>'), row=2, col=1)

# 4 – budget vs revenue scatter
sd = df[(df['budget_usd'] > 0) & (df['revenue_usd'] > 0)]
fig.add_trace(go.Scatter(x=sd['budget_usd']/1e6, y=sd['revenue_usd']/1e6, mode='markers',
                         marker=dict(color=sd['vote_average'], colorscale='Viridis',
                                     showscale=True, size=8, opacity=0.7,
                                     colorbar=dict(title='Rating', x=1.02, len=0.45, y=0.12)),
                         text=sd['title'], name='Budget/Revenue',
                         hovertemplate='%{text}<br>Budget: $%{x:.0f}M<br>Revenue: $%{y:.0f}M<extra></extra>'),
              row=2, col=2)

fig.update_layout(title_text='Movie Industry Analytics Dashboard', title_font_size=18,
                  template='plotly_white', height=700, width=1100, showlegend=False,
                  font=dict(family='Inter', size=11))

html_path = INTERACTIVE_DIR / 'interactive_dashboard.html'
fig.write_html(str(html_path))
fig.show()
print(f'Saved → {html_path}')

Saved → ../outputs/visualizations/interactive/interactive_dashboard.html


---
## Automated Chart Generation

The `chart_generator.py` module wraps all chart functions into a single `generate_all()` call.  
Running the script from the project root generates all 13 charts automatically.

In [18]:
# Demonstrate the automated generator
from visualization.chart_generator import generate_all
import os; os.chdir('..')  # move to project root so relative paths work
results = generate_all()
print('\nStatic charts saved:')
for name, paths in results['static'].items():
    print(f'  {name}: {paths["png"]}')
print('\nInteractive charts saved:')
for name, path in results['interactive'].items():
    print(f'  {name}: {path}')


  Lab 12 – Data Visualization Generator

Dataset: 126 movies, 15 columns

── Static charts (matplotlib / seaborn) ──
  [static]  top_movies_revenue
            PNG → outputs/visualizations/static/top_movies_by_revenue.png
            PDF → outputs/visualizations/static/top_movies_by_revenue.pdf
  [static]  avg_rating_over_years
            PNG → outputs/visualizations/static/avg_rating_over_years.png
            PDF → outputs/visualizations/static/avg_rating_over_years.pdf
  [static]  budget_vs_revenue
            PNG → outputs/visualizations/static/budget_vs_revenue_scatter.png
            PDF → outputs/visualizations/static/budget_vs_revenue_scatter.pdf
  [static]  rating_distribution
            PNG → outputs/visualizations/static/rating_distribution.png
            PDF → outputs/visualizations/static/rating_distribution.pdf
  [static]  genre_rating_boxplot
            PNG → outputs/visualizations/static/genre_rating_boxplot.png
            PDF → outputs/visualizations/static/genre

---
## Document Visualization Choices

This section explains **why each chart type was chosen** for each analysis question.
For every chart, the reasoning covers:
- What question it answers
- Why that chart type matches the statistical nature of the data
- What encoding decisions were made (colour, size, axis scale)


### Bar Chart (Horizontal) – Top 10 Movies by Revenue

The question is a **ranking comparison across named categories** (film titles). Horizontal bars are preferred over vertical bars when category labels are long strings, because they read left-to-right naturally without rotation. The `viridis` palette encodes rank via luminance — darker at the bottom, brighter at the top. Inline value annotations (`$X.XXB`) eliminate the need to scan the x-axis for every bar.

---

### Dual-Axis Line + Bar – Average Rating Over Years

Two related but differently scaled time series are displayed together: **movie count** (a count variable, encoded as bars on the left y-axis) and **average rating** (a continuous 0–10 variable, encoded as a line on the right y-axis). A dual-axis chart is justified here because both series share a temporal x-axis and a reader naturally wants to correlate them. `ax.twinx()` creates the second axis without a second figure.

---

### Scatter Plot – Budget vs Revenue

The central financial question is whether higher budgets yield higher revenue. A **scatter plot** is the canonical chart for two continuous variables where we want to see correlation, outliers, and clusters simultaneously. Genre is encoded as **hue** (categorical colour) to reveal whether genre moderates the budget–revenue relationship. Vote average is encoded as **marker size** because it is a secondary dimension. The diagonal break-even line makes profitability immediately visible — points above the line are profitable.

---

### Histogram + KDE – Rating Distribution

The question is about the **shape of a single continuous variable's distribution** — whether it is normal, skewed, or bi-modal. A histogram reveals raw frequency per bin; the KDE overlay smooths bin-boundary artefacts and shows the underlying density curve. Mean and median vertical reference lines highlight central tendency and any skew at a glance.

---

### Box-and-Whisker – Rating by Genre

When comparing **distributions of a continuous variable across many groups**, box plots are more space-efficient than overlaid histograms. They show the median, interquartile range (IQR), and outliers in a single compact glyph. Genres are sorted by median rating descending so the ranking is immediately readable. `hue='primary_genre'` with `legend=False` is the correct seaborn ≥ 0.12 pattern for colouring by the same column used on the x-axis.

---

### Correlation Heatmap

A correlation matrix is an N×N grid of pairwise statistics. A **heatmap** is the standard encoding: the diverging `coolwarm` palette maps negative correlations to cool blue and positive correlations to warm red, with white at zero. Annotating each cell with the numeric value prevents misreading from colour alone. `square=True` ensures every cell has equal visual weight regardless of figure dimensions.

---

### Vertical Bar – Genre Count

The question is a **frequency distribution across a discrete categorical variable**. Bar length is the most perceptually accurate channel for magnitude comparisons. Vertical orientation is used here because genre names are short and the chart is wider than it is tall. Inline count labels above each bar eliminate the need to read off the y-axis.

---

### 2×2 Multi-Panel Dashboard

An executive summary must convey several different questions at a glance. A **2×2 subplot grid** combines the four most informative individual views into a single shareable figure. The layout follows a natural reading order: top-left (revenue ranking), top-right (rating distribution), bottom-left (genre composition), bottom-right (budget/revenue relationship). `fig.tight_layout()` prevents panel labels from overlapping.

---

### Interactive Scatter (Plotly Express) – Budget vs Revenue

The static scatter plot raises a follow-up question: *'Which specific film is that outlier?'* Interactive **hover tooltips** answer it without overcrowding the chart with text labels. The Plotly version adds ROI to the tooltip so an analyst can assess the profitability of individual films. The legend is also clickable — genres can be hidden or isolated.

---

### Interactive Line – Movies per Year

A **time-series line chart** is the correct choice when the reader wants to track a single metric's evolution over time. Interactivity lets users zoom into specific decades and hover to see exact counts, average ratings, and total revenue for any individual year.

---

### Interactive Box Plot – Genre Ratings (Plotly)

The interactive version of the boxplot adds per-movie hover so the reader can identify which specific title is an outlier within a genre. `category_orders` keeps genres sorted by median, matching the static version. The legend is hidden (`showlegend=False`) because genre is already encoded on the x-axis.

---

### Interactive 2×2 Dashboard (Plotly Graph Objects)

The Plotly dashboard uses `make_subplots()` with `go` traces instead of Plotly Express because Express does not support multi-panel layouts. Each panel is independently zoomable and hoverable. The colorbar in Panel 4 encodes vote_average as a continuous colour scale, adding a third dimension to the budget/revenue scatter without adding a legend.
